In [1]:
from google.colab import files
uploaded = files.upload()

import numpy as np

# Load only the needed files
X_train = np.loadtxt("arcene_train.data")
y_train = np.loadtxt("arcene_train.labels")

X_valid = np.loadtxt("arcene_valid.data")
y_valid = np.loadtxt("arcene_valid.labels")

X_test = np.loadtxt("arcene_test.data")

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)


Saving arcene_valid.labels to arcene_valid.labels
Saving arcene_test.data to arcene_test.data
Saving arcene_train.data to arcene_train.data
Saving arcene_train.labels to arcene_train.labels
Saving arcene_valid.data to arcene_valid.data
Training: (100, 10000)
Validation: (100, 10000)
Test: (700, 10000)


In [2]:
# ================================
# TASK 1 — DATA PREPARATION
# ================================

from sklearn.preprocessing import MinMaxScaler

print("Task 1: Dataset Preparation")

# 1. Normalize using MinMaxScaler
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print("\nData Normalized Successfully!")
print("Training Scaled Shape:", X_train_scaled.shape)
print("Validation Scaled Shape:", X_valid_scaled.shape)
print("Test Scaled Shape:", X_test_scaled.shape)


Task 1: Dataset Preparation

Data Normalized Successfully!
Training Scaled Shape: (100, 10000)
Validation Scaled Shape: (100, 10000)
Test Scaled Shape: (700, 10000)


In [3]:
# ================================
# TASK 2 — PCA + FIRST SVM
# ================================

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report

print("Task 2: PCA + First SVM")

# 1. PCA dimensionality reduction
pca = PCA(n_components=100, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_valid_pca = pca.transform(X_valid_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("\nPCA Completed!")
print("PCA Training Shape:", X_train_pca.shape)

# 2. Train First SVM
svm1 = SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=42)
svm1.fit(X_train_pca, y_train)

# 3. Predict on validation
y_pred_svm1 = svm1.predict(X_valid_pca)

# 4. Classification report
print("\n===== FIRST SVM CLASSIFICATION REPORT =====\n")
print(classification_report(y_valid, y_pred_svm1))


Task 2: PCA + First SVM

PCA Completed!
PCA Training Shape: (100, 100)

===== FIRST SVM CLASSIFICATION REPORT =====

              precision    recall  f1-score   support

        -1.0       0.85      0.71      0.78        56
         1.0       0.70      0.84      0.76        44

    accuracy                           0.77       100
   macro avg       0.77      0.78      0.77       100
weighted avg       0.78      0.77      0.77       100



In [4]:
# ================================
# TASK 3 — BOOSTING SVM ARCHITECTURE
# ================================

print("Task 3: Boosting SVMs")

# STEP 1 — Identify misclassified samples
mis_idx = np.where(svm1.predict(X_train_pca) != y_train)[0]

# STEP 2 — Identify low-confidence samples
probs = svm1.predict_proba(X_train_pca)
conf = np.max(probs, axis=1)
low_conf_idx = np.argsort(conf)[:len(mis_idx)]

# Combine indices
boost_idx = np.unique(np.concatenate([mis_idx, low_conf_idx]))

X_boost = X_train_pca[boost_idx]
y_boost = y_train[boost_idx]

print("\nBoosting Dataset Size:", len(boost_idx))

# STEP 3 — Train Second SVM
svm2 = SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=42)
svm2.fit(X_boost, y_boost)

# STEP 4 — Boosting Prediction Logic
def boosting_predict(X):
    pred1 = svm1.predict(X)
    conf1 = svm1.predict_proba(X).max(axis=1)

    final = []
    for i in range(len(X)):
        if conf1[i] < 0.70:  # low confidence → send to second SVM
            final.append(svm2.predict(X[i].reshape(1, -1))[0])
        else:
            final.append(pred1[i])
    return np.array(final)

# STEP 5 — Evaluate Boosting Model
y_pred_boost = boosting_predict(X_valid_pca)

print("\n===== BOOSTING SVM CLASSIFICATION REPORT =====\n")
print(classification_report(y_valid, y_pred_boost))


Task 3: Boosting SVMs

Boosting Dataset Size: 8

===== BOOSTING SVM CLASSIFICATION REPORT =====

              precision    recall  f1-score   support

        -1.0       0.85      0.82      0.84        56
         1.0       0.78      0.82      0.80        44

    accuracy                           0.82       100
   macro avg       0.82      0.82      0.82       100
weighted avg       0.82      0.82      0.82       100

